## 引导文本转语音以生成更富表现力的音频

我们传统的 [TTS API](https://platform.openai.com/docs/guides/text-to-speech) 无法引导所生成音频的声音。例如，如果你想把一段文字转换为音频，就无法针对音频生成给出任何具体指令。

借助 [audio chat completions](https://platform.openai.com/docs/guides/audio/quickstart)，你可以在生成音频之前给出具体指令。这样你就可以让 API 以不同的语速、语调和口音说话。配合恰当的指令，这些声音可以更加生动、自然，也更贴合上下文。

### 传统 TTS

传统 TTS 可以指定声音，但无法指定语调、口音或任何其他与上下文相关的音频参数。

In [2]:
from openai import OpenAI
client = OpenAI()

tts_text = """
Once upon a time, Leo the lion cub woke up to the smell of pancakes and scrambled eggs.
His tummy rumbled with excitement as he raced to the kitchen. Mama Lion had made a breakfast feast!
Leo gobbled up his pancakes, sipped his orange juice, and munched on some juicy berries.
"""

speech_file_path = "./sounds/default_tts.mp3"
response = client.audio.speech.create(
    model="tts-1-hd",
    voice="alloy",
    input=tts_text,
)

response.write_to_file(speech_file_path)

### Chat Completions TTS

借助 chat completions，你可以在生成音频之前给出具体指令。在下面的示例中，我们在面向儿童的学习场景中生成英式口音。这对教育类应用尤其有用，因为助手的声音对学习体验很重要。

In [4]:
import base64

speech_file_path = "./sounds/chat_completions_tts.mp3"
completion = client.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "mp3"},
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that can generate audio from text. Speak in a British accent and enunciate like you're talking to a child.",
        },
        {
            "role": "user",
            "content": tts_text,
        }
    ],
)

mp3_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open(speech_file_path, "wb") as f:
    f.write(mp3_bytes)

speech_file_path = "./sounds/chat_completions_tts_fast.mp3"
completion = client.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "mp3"},
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that can generate audio from text. Speak in a British accent and speak really fast.",
        },
        {
            "role": "user",
            "content": tts_text,
        }
    ],
)

mp3_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open(speech_file_path, "wb") as f:
    f.write(mp3_bytes)

### Chat Completions 多语言 TTS

我们还可以生成带有不同语言口音的音频。在下面的示例中，我们生成带有特定乌拉圭西班牙语口音的音频。

In [12]:
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are an expert translator. Translate any text given into Spanish like you are from Uruguay.",
        },
        {
            "role": "user",
            "content": tts_text,
        }
    ],
)
translated_text = completion.choices[0].message.content
print(translated_text)

speech_file_path = "./sounds/chat_completions_tts_es_uy.mp3"
completion = client.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "mp3"},
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that can generate audio from text. Speak any text that you receive in a Uruguayan spanish accent and more slowly.",
        },
        {
            "role": "user",
            "content": translated_text,
        }
    ],
)

mp3_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open(speech_file_path, "wb") as f:
    f.write(mp3_bytes)

Había una vez un leoncito llamado Leo que se despertó con el aroma de panqueques y huevos revueltos. Su pancita gruñía de emoción mientras corría hacia la cocina. ¡Mamá León había preparado un festín de desayuno! Leo devoró sus panqueques, sorbió su jugo de naranja y mordisqueó algunas bayas jugosas.


## 总结

能够引导所生成音频的声音，为打造更丰富的音频体验开辟了诸多可能。用例有很多，例如：
- **更强的表现力**：可引导的 TTS 允许调整语调、音高、语速和情绪，使声音能够传达不同的情绪（例如兴奋、平静、紧迫）。
- **语言学习与教育**：可引导的 TTS 可以模仿口音、语调变化和发音，这对语言学习者以及对准确语调和重音要求很高的教育类应用很有帮助。
- **贴合上下文的声音**：可引导的 TTS 能让声音适应内容的上下文，例如专业文档使用正式语气，社交互动使用友好、对话式的风格。这有助于在虚拟助手和聊天机器人中营造更自然的对话。
